# 🎯 Orchestrator — Full Pipeline Runner
## Runs all 4 agents end-to-end in one shot

```
input_schema.json
  → Agent 1 (Prompt Engineer)  → scripts/ + manifest
  → Agent 2 (Scraper Runner)   → data/raw/*.json
  → Agent 3 (Analyst)          → data/analysed/full_analysis.json
  → Agent 4 (Report Writer)    → reports/competitive_report.html
```

**Edit `input_schema.json` then Run All Cells.**

In [9]:
import os, json, subprocess, sys, re
import urllib.request
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import anthropic

load_dotenv()
print('ANTHROPIC_API_KEY loaded:', bool(os.getenv('ANTHROPIC_API_KEY')))

BASE_DIR      = Path('.')
SCRIPTS_DIR   = BASE_DIR / 'scripts'
DATA_RAW      = BASE_DIR / 'data' / 'raw'
DATA_ANALYSED = BASE_DIR / 'data' / 'analysed'
DATA_LOGS     = BASE_DIR / 'data' / 'logs'
REPORTS_DIR   = BASE_DIR / 'reports'

for d in [SCRIPTS_DIR, DATA_RAW, DATA_ANALYSED, DATA_LOGS, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

with open('input_schema.json') as f:
    INPUT = json.load(f)

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from .env

print(f'Business   : {INPUT["business"]["name"]}')
print(f'Competitors: {[c["name"] for c in INPUT["competitors"]]}')

ANTHROPIC_API_KEY loaded: True
Business   : PuneLaundry
Competitors: ['UClean Pune', 'Laundry Basket', 'Washmart', 'Mr. Fresh', 'Fabcure']


## All Tools — Agents 1–4

In [10]:
# ── AGENT 1 TOOLS ──────────────────────────────────
def get_business_context() -> str:
    return json.dumps(INPUT, indent=2)

def get_playwright_boilerplate() -> str:
    return ('async playwright, random delays 1.5-3.5s, '
            'networkidle, try/except per section, screenshot at end. '
            'Save to data/raw/{name}_raw.json. '
            'Fields: competitor_name, website, scraped_at, scrape_status, '
            'pricing, features, homepage_headline, homepage_usp, '
            'reviews_summary, blog_topics, job_postings, errors')

def save_generated_script(competitor_name: str, script_code: str) -> str:
    safe = competitor_name.lower().replace(' ', '_')
    p = SCRIPTS_DIR / f'scrape_{safe}.py'
    p.write_text(script_code)
    print(f'  Saved: {p.name}')
    return str(p)

def validate_python_syntax(script_code: str) -> str:
    import ast
    try:
        ast.parse(script_code); return 'valid'
    except SyntaxError as e:
        return f'SyntaxError line {e.lineno}: {e.msg}'

def save_scrape_manifest(manifest: str) -> str:
    p = BASE_DIR / 'scrape_manifest.json'
    p.write_text(manifest)
    print('  Manifest saved')
    return str(p)

AGENT1_TOOLS = [
    {"name": "get_business_context", "description": "Returns full business context from input_schema.json.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "get_playwright_boilerplate", "description": "Returns Playwright coding standards and field spec.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "save_generated_script", "description": "Saves a Playwright script file for a competitor.",
     "input_schema": {"type": "object", "properties": {
         "competitor_name": {"type": "string"}, "script_code": {"type": "string"}},
         "required": ["competitor_name", "script_code"]}},
    {"name": "validate_python_syntax", "description": "Validates Python syntax. Returns 'valid' or error.",
     "input_schema": {"type": "object", "properties": {
         "script_code": {"type": "string"}}, "required": ["script_code"]}},
    {"name": "save_scrape_manifest", "description": "Saves scrape manifest JSON string to disk.",
     "input_schema": {"type": "object", "properties": {
         "manifest": {"type": "string"}}, "required": ["manifest"]}},
]

AGENT1_FNS = {
    "get_business_context": lambda **k: get_business_context(),
    "get_playwright_boilerplate": lambda **k: get_playwright_boilerplate(),
    "save_generated_script": lambda **k: save_generated_script(**k),
    "validate_python_syntax": lambda **k: validate_python_syntax(**k),
    "save_scrape_manifest": lambda **k: save_scrape_manifest(**k),
}

print('Agent 1 tools loaded')

Agent 1 tools loaded


In [11]:
# ── AGENT 2 TOOLS ──────────────────────────────────
def get_manifest() -> str:
    p = BASE_DIR / 'scrape_manifest.json'
    return p.read_text() if p.exists() else json.dumps({'error': 'No manifest'})

def run_scraper_script(competitor_name: str, script_path: str) -> str:
    script = Path(script_path)
    if not script.exists():
        return json.dumps({'status': 'failed', 'error': 'Script not found'})
    try:
        print(f'  Scraping {competitor_name}...')
        proc = subprocess.run([sys.executable, str(script)], capture_output=True, text=True, timeout=120)
        out = DATA_RAW / f'{competitor_name.lower().replace(" ","_")}_raw.json'
        if proc.returncode == 0 and out.exists():
            return json.dumps({'status': 'success', 'output': str(out)})
        return json.dumps({'status': 'failed', 'stderr': proc.stderr[-300:]})
    except subprocess.TimeoutExpired:
        return json.dumps({'status': 'timeout'})
    except Exception as e:
        return json.dumps({'status': 'error', 'error': str(e)})

def run_fallback_scraper(competitor_name: str, url: str) -> str:
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=15) as r:
            html = r.read().decode('utf-8', errors='ignore')
        h1s = [re.sub(r'<[^>]+>', '', h).strip()
               for h in re.findall(r'<h1[^>]*>(.*?)</h1>', html, re.I|re.S)][:3]
        data = {'competitor_name': competitor_name, 'website': url,
                'scraped_at': datetime.now().isoformat(), 'scrape_status': 'partial_fallback',
                'homepage_headline': h1s[0] if h1s else '', 'homepage_usp': h1s,
                'pricing': {}, 'features': [], 'reviews_summary': {},
                'blog_topics': [], 'job_postings': {}, 'errors': ['Used HTTP fallback']}
        out = DATA_RAW / f'{competitor_name.lower().replace(" ","_")}_raw.json'
        out.write_text(json.dumps(data, indent=2))
        return json.dumps({'status': 'partial_fallback', 'output': str(out)})
    except Exception as e:
        return json.dumps({'status': 'failed', 'error': str(e)})

def check_scrape_output(competitor_name: str) -> str:
    f = DATA_RAW / f'{competitor_name.lower().replace(" ","_")}_raw.json'
    if not f.exists(): return json.dumps({'exists': False})
    d = json.loads(f.read_text())
    return json.dumps({'exists': True, 'status': d.get('scrape_status'),
                       'has_pricing': bool(d.get('pricing')), 'has_features': bool(d.get('features'))})

def save_scrape_log(log_data: str) -> str:
    p = DATA_LOGS / 'scrape_latest.json'
    p.write_text(log_data)
    return str(p)

AGENT2_TOOLS = [
    {"name": "get_manifest", "description": "Returns the scrape manifest JSON.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "run_scraper_script", "description": "Runs a Playwright scraper script for a competitor.",
     "input_schema": {"type": "object", "properties": {
         "competitor_name": {"type": "string"}, "script_path": {"type": "string"}},
         "required": ["competitor_name", "script_path"]}},
    {"name": "run_fallback_scraper", "description": "HTTP fallback scraper when Playwright fails.",
     "input_schema": {"type": "object", "properties": {
         "competitor_name": {"type": "string"}, "url": {"type": "string"}},
         "required": ["competitor_name", "url"]}},
    {"name": "check_scrape_output", "description": "Validates the scrape output JSON file for a competitor.",
     "input_schema": {"type": "object", "properties": {
         "competitor_name": {"type": "string"}}, "required": ["competitor_name"]}},
    {"name": "save_scrape_log", "description": "Saves scrape run log JSON.",
     "input_schema": {"type": "object", "properties": {
         "log_data": {"type": "string"}}, "required": ["log_data"]}},
]

AGENT2_FNS = {
    "get_manifest": lambda **k: get_manifest(),
    "run_scraper_script": lambda **k: run_scraper_script(**k),
    "run_fallback_scraper": lambda **k: run_fallback_scraper(**k),
    "check_scrape_output": lambda **k: check_scrape_output(**k),
    "save_scrape_log": lambda **k: save_scrape_log(**k),
}

print('Agent 2 tools loaded')

Agent 2 tools loaded


In [12]:
# ── AGENT 3 TOOLS ──────────────────────────────────
def _load_raw():
    d = {}
    for f in DATA_RAW.glob('*_raw.json'):
        try:
            data = json.loads(f.read_text())
            d[data.get('competitor_name', f.stem)] = data
        except: pass
    return d

def get_all_raw_data() -> str:
    return json.dumps({'our_business': INPUT['business'], 'competitors': _load_raw()}, indent=2)

def build_pricing_comparison() -> str:
    raw = _load_raw()
    t = [{'competitor': n, 'per_kg': d.get('pricing',{}).get('per_kg','Unknown'),
          'express_surcharge': d.get('pricing',{}).get('express_surcharge','Unknown')} for n,d in raw.items()]
    return json.dumps({'pricing': t, 'ours': 'Transparent per-kg pricing, same-day delivery'}, indent=2)

def build_feature_matrix() -> str:
    raw = _load_raw()
    feats = ['Pickup & Delivery','Same-day Express','Dry Cleaning','Ironing',
             'Shoe Care','Subscription Plan','App Booking','Web Booking',
             'Eco-friendly Wash','Loyalty Program','GST Invoice','Pune Coverage']
    matrix = {n: {f: any(kw in ' '.join(str(x) for x in d.get('features',[])).lower()
                         for kw in f.lower().split()) for f in feats}
              for n,d in raw.items()}
    matrix[INPUT['business']['name']] = {f: f in ['Pickup & Delivery','Same-day Express',
        'Dry Cleaning','Ironing','App Booking','Web Booking','Eco-friendly Wash','GST Invoice','Pune Coverage'] for f in feats}
    return json.dumps({'matrix': matrix, 'features': feats}, indent=2)

def analyze_messaging_positioning() -> str:
    raw = _load_raw()
    return json.dumps({'messaging': [{'competitor': n, 'headline': d.get('homepage_headline',''),
                                       'usps': d.get('homepage_usp',[])} for n,d in raw.items()]}, indent=2)

def analyze_content_and_jobs() -> str:
    raw = _load_raw()
    return json.dumps({'signals': [{'competitor': n, 'blog': d.get('blog_topics',[])[:5],
                                    'jobs': d.get('job_postings',{}).get('total',0)} for n,d in raw.items()]}, indent=2)

def identify_gaps_and_opportunities() -> str:
    return json.dumps({
        'pricing_gaps': ['No competitor offers transparent per-kg online calculator','Subscription plans underutilized'],
        'service_gaps': ['Shoe care widely missing','Curtain/sofa cleaning niche'],
        'coverage_gaps': ['Hadapsar and Pune Camp underserved'],
        'opportunities': ['Corporate tie-ups with IT parks in Hinjewadi','WhatsApp booking flow','Eco-friendly certification badge']
    }, indent=2)

def score_competitors() -> str:
    raw = _load_raw()
    scores = []
    for n,d in raw.items():
        fc = len(d.get('features',[])); rr = d.get('reviews_summary',{}).get('rating',3.5)
        jc = d.get('job_postings',{}).get('total',0)
        s = round((min(10,fc/2)+round(rr*2,1)+min(10,jc/10))/3,1)
        scores.append({'competitor': n, 'score': s, 'level': 'high' if s>=7 else ('medium' if s>=5 else 'low')})
    return json.dumps({'scores': sorted(scores, key=lambda x: x['score'], reverse=True)}, indent=2)

def save_analysis_outputs(analysis_json: str) -> str:
    p = DATA_ANALYSED / 'full_analysis.json'
    p.write_text(analysis_json)
    print('  Analysis saved')
    return str(p)

AGENT3_TOOLS = [
    {"name": "get_all_raw_data", "description": "Returns all raw competitor scraped data.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "build_pricing_comparison", "description": "Builds per-kg pricing comparison table.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "build_feature_matrix", "description": "Creates laundry service feature comparison matrix.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "analyze_messaging_positioning", "description": "Analyzes competitor homepage messaging and USPs.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "analyze_content_and_jobs", "description": "Analyzes blog topics and hiring signals.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "identify_gaps_and_opportunities", "description": "Identifies strategic gaps and opportunities for PuneLaundry.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "score_competitors", "description": "Scores competitors on threat level.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "save_analysis_outputs", "description": "Saves full analysis JSON to disk.",
     "input_schema": {"type": "object", "properties": {
         "analysis_json": {"type": "string"}}, "required": ["analysis_json"]}},
]

AGENT3_FNS = {
    "get_all_raw_data": lambda **k: get_all_raw_data(),
    "build_pricing_comparison": lambda **k: build_pricing_comparison(),
    "build_feature_matrix": lambda **k: build_feature_matrix(),
    "analyze_messaging_positioning": lambda **k: analyze_messaging_positioning(),
    "analyze_content_and_jobs": lambda **k: analyze_content_and_jobs(),
    "identify_gaps_and_opportunities": lambda **k: identify_gaps_and_opportunities(),
    "score_competitors": lambda **k: score_competitors(),
    "save_analysis_outputs": lambda **k: save_analysis_outputs(**k),
}

print('Agent 3 tools loaded')

Agent 3 tools loaded


In [13]:
# ── AGENT 4 TOOLS ──────────────────────────────────
def get_full_analysis() -> str:
    p = DATA_ANALYSED / 'full_analysis.json'
    return p.read_text() if p.exists() else json.dumps({'error': 'No analysis'})

def get_report_template() -> str:
    return ('10 sections: Header, Exec Summary, Threat Scores, Pricing Table, '
            'Feature Matrix, Messaging, Signals, Gaps, Recommendations, Footer. '
            'Dark theme #060d1f. Inline CSS. Checkmarks for features. Real data only. '
            'Business: PuneLaundry — laundry services in Pune, India.')

def generate_html_report(html_content: str) -> str:
    ts = datetime.now().strftime('%Y%m%d_%H%M')
    p = REPORTS_DIR / f'competitive_report_{ts}.html'
    p.write_text(html_content, encoding='utf-8')
    (REPORTS_DIR / 'competitive_report_latest.html').write_text(html_content, encoding='utf-8')
    print(f'  HTML saved ({p.stat().st_size:,} bytes)')
    return str(p)

def export_pdf_report(html_path: str) -> str:
    try:
        from weasyprint import HTML
        pdf = html_path.replace('.html', '.pdf')
        HTML(filename=html_path).write_pdf(pdf)
        print('  PDF exported')
        return pdf
    except Exception as e:
        return f'PDF skipped: {e}'

def save_executive_summary(summary_markdown: str) -> str:
    p = REPORTS_DIR / 'executive_summary.md'
    p.write_text(summary_markdown)
    print('  Executive summary saved')
    return str(p)

AGENT4_TOOLS = [
    {"name": "get_full_analysis", "description": "Returns the full competitor analysis JSON.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "get_report_template", "description": "Returns report structure and styling guidelines.",
     "input_schema": {"type": "object", "properties": {}, "required": []}},
    {"name": "generate_html_report", "description": "Saves complete HTML competitive intelligence report to disk.",
     "input_schema": {"type": "object", "properties": {
         "html_content": {"type": "string"}}, "required": ["html_content"]}},
    {"name": "export_pdf_report", "description": "Exports PDF from an HTML report file path.",
     "input_schema": {"type": "object", "properties": {
         "html_path": {"type": "string"}}, "required": ["html_path"]}},
    {"name": "save_executive_summary", "description": "Saves executive summary as markdown.",
     "input_schema": {"type": "object", "properties": {
         "summary_markdown": {"type": "string"}}, "required": ["summary_markdown"]}},
]

AGENT4_FNS = {
    "get_full_analysis": lambda **k: get_full_analysis(),
    "get_report_template": lambda **k: get_report_template(),
    "generate_html_report": lambda **k: generate_html_report(**k),
    "export_pdf_report": lambda **k: export_pdf_report(**k),
    "save_executive_summary": lambda **k: save_executive_summary(**k),
}

print('Agent 4 tools loaded')

Agent 4 tools loaded


## Agentic Loop Helper (Claude)

In [14]:
def run_claude_agent(label, model, system, tools, tool_fns, prompt, max_tokens=8192):
    """
    Runs a Claude agentic loop with tool use.
    Loops until stop_reason == 'end_turn', executing tools on each 'tool_use' stop.
    Returns the final text response.
    """
    messages = [{"role": "user", "content": prompt}]
    iteration = 0

    while True:
        iteration += 1
        response = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            system=system,
            tools=tools,
            messages=messages
        )

        if response.stop_reason == "end_turn":
            final_text = next((b.text for b in response.content if hasattr(b, 'text')), '')
            return final_text

        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    fn = tool_fns.get(block.name)
                    try:
                        result = fn(**(block.input or {})) if fn else f"Unknown tool: {block.name}"
                    except Exception as e:
                        result = f"Tool error: {e}"
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            messages.append({"role": "user", "content": tool_results})
        else:
            # Unexpected stop reason
            return f"Stopped: {response.stop_reason}"

# Agent configs — Claude Opus 4.8 for reasoning agents, Haiku 4.5 for the fast scraper runner
AGENTS = [
    {
        "label": "Agent 1 — Prompt Engineer",
        "model": "claude-opus-4-8",
        "system": (
            "You are a Playwright script generator for competitive intelligence. "
            "Use tools in order: get_business_context → get_playwright_boilerplate → "
            "for each competitor: validate_python_syntax → save_generated_script → "
            "finally save_scrape_manifest. "
            "Scripts must scrape: pricing (per-kg rates), features (services offered), "
            "homepage headline, USP, reviews, turnaround time, service areas, app ratings."
        ),
        "tools": AGENT1_TOOLS,
        "fns": AGENT1_FNS,
        "prompt": "Generate complete async Playwright scraping scripts for all competitors in input_schema.json. Save each script and the manifest."
    },
    {
        "label": "Agent 2 — Scraper Runner",
        "model": "claude-haiku-4-5-20251001",
        "system": (
            "You are a scraper executor. "
            "Use tools in order: get_manifest → for each competitor: run_scraper_script → "
            "if failed/timeout: run_fallback_scraper → check_scrape_output → save_scrape_log. "
            "Always attempt fallback before giving up on a competitor."
        ),
        "tools": AGENT2_TOOLS,
        "fns": AGENT2_FNS,
        "prompt": "Execute all scrapers from the manifest. Use HTTP fallback for any failures. Validate outputs and save the run log."
    },
    {
        "label": "Agent 3 — Analyst",
        "model": "claude-opus-4-8",
        "system": (
            "You are a competitive intelligence analyst specialising in laundry services in Pune, India. "
            "Use tools in order: get_all_raw_data → build_pricing_comparison → build_feature_matrix → "
            "analyze_messaging_positioning → analyze_content_and_jobs → identify_gaps_and_opportunities → "
            "score_competitors → compile a master JSON with keys: pricing, features, messaging, signals, "
            "gaps, scores, executive_summary (3 sentences), top_3_recommendations → save_analysis_outputs."
        ),
        "tools": AGENT3_TOOLS,
        "fns": AGENT3_FNS,
        "prompt": "Analyse all competitor data and produce a comprehensive competitive intelligence JSON. Save it via save_analysis_outputs."
    },
    {
        "label": "Agent 4 — Report Writer",
        "model": "claude-opus-4-8",
        "system": (
            "You are a competitive intelligence report designer. "
            "Use tools in order: get_full_analysis → get_report_template → "
            "generate a complete, self-contained HTML report with all 10 sections, "
            "dark theme (#060d1f background), inline CSS, real data from analysis — "
            "NO placeholder text → generate_html_report → export_pdf_report → save_executive_summary."
        ),
        "tools": AGENT4_TOOLS,
        "fns": AGENT4_FNS,
        "prompt": "Generate the complete PuneLaundry competitive intelligence HTML report with all 10 sections and executive summary."
    },
]

print('All 4 Claude agents configured (Opus 4.8 + Haiku 4.5)')

All 4 Claude agents configured (Opus 4.8 + Haiku 4.5)


## 🚀 Run Full Pipeline

In [15]:
pipeline_start = datetime.now()

for agent_cfg in AGENTS:
    print(f'\n{"="*60}')
    print(f'Running: {agent_cfg["label"]}  [{agent_cfg["model"]}]')
    print(f'{"="*60}')
    t0 = datetime.now()

    final_output = run_claude_agent(
        label=agent_cfg["label"],
        model=agent_cfg["model"],
        system=agent_cfg["system"],
        tools=agent_cfg["tools"],
        tool_fns=agent_cfg["fns"],
        prompt=agent_cfg["prompt"]
    )

    elapsed = (datetime.now() - t0).seconds
    print(f'Done in {elapsed}s')
    print(final_output[:400] if final_output else '(no text output)')

total = (datetime.now() - pipeline_start).seconds
print(f'\n{"="*60}')
print(f'PIPELINE COMPLETE in {total}s')


Running: Agent 1 — Prompt Engineer  [claude-opus-4-8]


AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CcbP38g55wYa7MraitJTG'}

## Final Output

In [ ]:
import webbrowser

print('Output files:')
for folder, label in [(SCRIPTS_DIR,'Scripts'),(DATA_RAW,'Raw Data'),
                       (DATA_ANALYSED,'Analysis'),(REPORTS_DIR,'Reports')]:
    files = list(folder.iterdir()) if folder.exists() else []
    print(f'  [{label}]: {len(files)} files')
    for f in sorted(files):
        print(f'    {f.name} ({f.stat().st_size:,} bytes)')

html = REPORTS_DIR / 'competitive_report_latest.html'
if html.exists():
    webbrowser.open(f'file://{html.resolve()}')
    print('\nReport opened in browser!')

em = REPORTS_DIR / 'executive_summary.md'
if em.exists():
    print('\n' + '='*60)
    print(em.read_text())

Output files:
  [Scripts]: 8 files
    scrape_cashfree.py (3,128 bytes)
    scrape_fabcure.py (1,476 bytes)
    scrape_laundry_basket.py (1,523 bytes)
    scrape_mr._fresh.py (1,482 bytes)
    scrape_payu.py (3,073 bytes)
    scrape_razorpay.py (3,138 bytes)
    scrape_uclean_pune.py (1,494 bytes)
    scrape_washmart.py (1,483 bytes)
  [Raw Data]: 5 files
    fabcure_raw.json (606 bytes)
    laundry_basket_raw.json (631 bytes)
    mr_fresh_raw.json (608 bytes)
    uclean_pune_raw.json (368 bytes)
    washmart_raw.json (469 bytes)
  [Analysis]: 1 files
    full_analysis.json (1,701 bytes)
  [Reports]: 6 files
    competitive_report_20260630_1242.html (2,949 bytes)
    competitive_report_20260630_2314.html (4,776 bytes)
    competitive_report_20260630_2326.html (3,794 bytes)
    competitive_report_latest.html (3,794 bytes)
    competitive_report_redesigned.html (15,743 bytes)
    executive_summary.md (322 bytes)

Report opened in browser!

# Executive Summary

The comprehensive analysis 